In [ ]:
# Parameters cell in Jupyter Notebook
stats_category = 'default_category'  # This default value will be replaced by Papermill
identifier = None  # Default value, can be empty if not required

In [ ]:
# Load environment variables
import dotenv
dotenv.load_dotenv('.env')

# Extract connection info
import os
mongodb_uri = os.getenv('MONGODB_URI')
postgres_host = os.getenv('POSTGRES_HOST')
postgres_port = os.getenv('POSTGRES_PORT')
postgres_db = os.getenv('POSTGRES_DB')
postgres_user = os.getenv('POSTGRES_USER')
postgres_password = os.getenv('POSTGRES_PASSWORD')

# Connect to MongoDB
from pymongo import MongoClient
mongo_client = MongoClient(mongodb_uri)
db = mongo_client['fdpdata']

# Connect to PostgreSQL
from sqlalchemy import create_engine
engine = create_engine(f'postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}')


# Import commonly used libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


## if identified is not null then cast as int
if identifier is not None:
    identifier = int(identifier)


# Players

## All Players

In [ ]:
# Cell: All Players Analysis
if stats_category == 'allPlayers':

    # Ensure the output directory exists
    output_dir = './outputs/allPlayers'
    os.makedirs(output_dir, exist_ok=True)

    # Load data
    players_data = pd.read_sql("SELECT * from players", engine)
    appearances_collection = db.appearances

    # Aggregating total goals by player
    appearances_pipeline = [
        {
            '$group': {
                '_id': '$player_id',
                'total_goals': {
                    '$sum': '$goals'
                }
            }
        }
    ]
    appearances_data = list(appearances_collection.aggregate(appearances_pipeline))
    appearances_df = pd.DataFrame(appearances_data)
    appearances_df.rename(columns={'_id': 'player_id'}, inplace=True)

    # Merging data
    players_data = players_data.merge(appearances_df, on='player_id', how='left')

    # Standard Statistics

    # Age Distribution Analysis
    plt.figure(figsize=(10, 6))
    sns.histplot(players_data['age'], bins=20, kde=True)
    plt.title('Age Distribution of Players')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.savefig(os.path.join(output_dir, 'age_distribution.svg'))

    # Positional Analysis
    positions_count = players_data['position'].value_counts()
    plt.figure(figsize=(10, 6))
    positions_count.plot(kind='bar')
    plt.title('Player Count by Position')
    plt.xlabel('Position')
    plt.ylabel('Count')
    plt.savefig(os.path.join(output_dir, 'player_count_by_position.svg'))

    # Club Affiliation Diversity - Top 10
    plt.figure(figsize=(10, 6))
    top_clubs = players_data['current_club_name'].value_counts().head(10)
    top_clubs.plot(kind='bar')
    plt.title('Top 10 Clubs by Historical Player Count')
    plt.xlabel('Club')
    plt.ylabel('Player Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'top_club_affiliation_diversity.svg'))


    # Expert Statistics

    # Market Value Analysis
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=players_data, x='position', y='market_value_in_eur',
                whis=[5, 95], showfliers=False)
    plt.title('Market Value by Position', fontsize=16)
    plt.xlabel('Position', fontsize=14)
    plt.ylabel('Market Value (EUR)', fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'market_value_by_position.svg'))

    # Physical Attributes Correlation
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=players_data, x='height_in_cm', y='market_value_in_eur')
    plt.title('Height vs Market Value')
    plt.xlabel('Height (cm)')
    plt.ylabel('Market Value (EUR)')
    plt.savefig(os.path.join(output_dir, 'height_vs_market_value.svg'))

    # International Representation - Top 10
    plt.figure(figsize=(10, 6))
    top_countries = players_data['country_of_citizenship'].value_counts().head(10)
    top_countries.plot(kind='bar')
    plt.title('Top 10 Countries by Player Count')
    plt.xlabel('Country')
    plt.xticks(rotation=45)
    plt.ylabel('Player Count')
    plt.tight_layout() 
    plt.savefig(os.path.join(output_dir, 'top_international_representation.svg'))

    # Performance and Age Relationship
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=players_data, x='age', y='total_goals')
    plt.title('Performance vs Age')
    plt.xlabel('Age')
    plt.ylabel('Total Goals')
    plt.savefig(os.path.join(output_dir, 'performance_vs_age.svg'))



## Single Player

In [ ]:
# Cell: Single Player Analysis
if stats_category == 'player' and identifier:
    
    # Ensure the output directory exists
    output_dir = f'./outputs/{identifier}'
    os.makedirs(output_dir, exist_ok=True)
    
    # Load player data from PostgreSQL
    player_query = f"SELECT * FROM players WHERE player_id = {identifier}"
    player_data = pd.read_sql(player_query, engine)

    # Load valuation data from MongoDB
    valuations_pipeline = [
        {
            '$match': {
                'player_id': identifier  # Ensure you are matching the type (e.g., integer)
            }
        },
        {
            '$project': {
                '_id': 0,
                'player_id': 1,
                'market_value_in_eur': 1,
                'date': { '$dateToString': { 'format': "%Y-%m-%d", 'date': "$date" } }  # Convert date to string
            }
        },
        {
            '$sort': {
                'date': 1  # Sort by date to get a chronological order
            }
        }
    ]

    valuations_collection = db.player_valuations
    valuations_data = list(valuations_collection.aggregate(valuations_pipeline))
    valuations_df = pd.DataFrame(valuations_data)


    # Merging data to include player info with valuations
    player_valuations = pd.merge(player_data, valuations_df, on='player_id', how='left')

    # Create the visualization
    plt.figure(figsize=(10, 6))
    plt.plot(player_valuations['date'], player_valuations['market_value_in_eur_y'], marker='o')
    plt.title(f"Market Value Over Time for {player_data['name'][0]}")
    plt.xlabel('Date')
    plt.ylabel('Market Value (MM EUR)')
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'market_value_{identifier}.svg'))


    # Load appearances data from MongoDB
    appearances_pipeline = [
        {
            '$match': {
                'player_id': identifier
            }
        },
        {
            '$lookup': {
                'from': 'games',
                'localField': 'game_id',
                'foreignField': 'game_id',
                'as': 'game_info'
            }
        },
        {
            '$unwind': '$game_info'
        },
        {
            '$group': {
                '_id': '$game_info.season',
                'total_goals': {'$sum': '$goals'},
                'total_assists': {'$sum': '$assists'},
                'appearances': {'$sum': 1},
                'yellow_cards': {'$sum': '$yellow_cards'},
                'red_cards': {'$sum': '$red_cards'}
            }
        },
        {
            '$sort': {'_id': 1}
        }
    ]

    appearances_data = list(db.appearances.aggregate(appearances_pipeline))
    performance_df = pd.DataFrame(appearances_data)

    # Create the visualization for Performance over Seasons and Disciplinary Record
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Performance data
    ax1.set_title(f"Performance and Disciplinary Record over Seasons for {player_data['name'][0]}")
    ax1.plot(performance_df['_id'], performance_df['total_goals'], label='Total Goals', marker='o', color='blue')
    ax1.plot(performance_df['_id'], performance_df['total_assists'], label='Total Assists', marker='o', color='green')
    ax1.set_xlabel('Season')
    ax1.set_ylabel('Performance Metrics', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # Disciplinary data
    ax2 = ax1.twinx()
    ax2.plot(performance_df['_id'], performance_df['yellow_cards'], label='Yellow Cards', marker='o', color='yellow')
    ax2.plot(performance_df['_id'], performance_df['red_cards'], label='Red Cards', marker='o', color='red')
    ax2.set_ylabel('Card Count', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # Legends and layout
    fig.tight_layout()
    fig.legend(loc='upper left', bbox_to_anchor=(0,1), bbox_transform=ax1.transAxes)
    
    # Save and show the visualization
    plt.savefig(os.path.join(output_dir, f'performance_disciplinary_{identifier}.svg'))
    plt.show()

    print(f"Performed analysis for player {identifier}")


    # Advanced Analysis

    # 1. Comparison with Averages
    # We will compare the player's total goals and assists with the league average.

    league_averages_pipeline = [
        {
            '$group': {
                '_id': '$season',
                'avg_goals': {'$avg': '$goals'},
                'avg_assists': {'$avg': '$assists'}
            }
        },
        {
            '$sort': {'_id': 1}
        }
    ]

    league_averages_data = list(db.appearances.aggregate(league_averages_pipeline))
    league_averages_df = pd.DataFrame(league_averages_data)

    # Visualization for Comparison with Averages
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.set_title(f"Player vs League Averages for {player_data['name'][0]}")
    ax1.plot(performance_df['_id'], performance_df['total_goals'], label='Player Total Goals', marker='o', color='blue')
    ax1.plot(league_averages_df['_id'], league_averages_df['avg_goals'], label='League Avg Goals', marker='^', color='green', linestyle='--')
    ax1.set_xlabel('Season')
    ax1.set_ylabel('Goals', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    ax2 = ax1.twinx()
    ax2.plot(performance_df['_id'], performance_df['total_assists'], label='Player Total Assists', marker='o', color='red')
    ax2.plot(league_averages_df['_id'], league_averages_df['avg_assists'], label='League Avg Assists', marker='^', color='purple', linestyle='--')
    ax2.set_ylabel('Assists', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    fig.tight_layout()
    fig.legend(loc='upper left', bbox_to_anchor=(0,1), bbox_transform=ax1.transAxes)
    plt.savefig(os.path.join(output_dir, f'player_vs_league_averages_{identifier}.svg'))
    plt.show()


    print(f"Advanced analysis completed for player {identifier}")



# Clubs

## All Clubs

In [ ]:
# Cell: All Clubs Analysis
if stats_category == 'allClubs':
    # Standard Statistics
    # Average Age per Club
    # Squad Size Comparison
    # Net Transfer Records
    # Foreigners Percentage

    # Expert Statistics
    # Club Performance Over Seasons
    # Transfer Record Trends
    # Financial Analysis

    # (Analysis code goes here)
    print("Performed All Clubs analysis")


## Single Club

In [ ]:
# Cell: Single Club Analysis
if stats_category == 'club' and identifier:
    # Standard Statistics
    # Season Performance
    # Top Scorers and Assist Providers
    # Transfer Activity Overview

    # Expert Statistics
    # Tactical Analysis
    # Performance in Different Competitions
    # Financial Analysis

    # (Analysis code goes here)
    print(f"Performed analysis for club {identifier}")


# Competitions

# All Competitions

In [ ]:
# Cell: All Competitions Analysis
if stats_category == 'allCompetitions':
    # Standard Statistics
    # Team Performance in Latest Season
    # Top Goal Scorers
    # Attendance Trends

    # Expert Statistics
    # Historical Trends in Competition
    # Competition Popularity
    # Referee Analysis

    # (Analysis code goes here)
    print("Performed All Competitions analysis")


## Single Competition